In [1]:
import tensorflow as tf
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D


In [3]:
%pip install tensorflow-datasets

  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Getting requirements to build wheel: started
  Getting requirements to build wheel: finished with status 'done'
  Preparing metadata (pyproject.toml): started
  Preparing metadata (pyproject.toml): finished with status 'done'
   ---------------------------------------- 0.0/5.3 MB ? eta -:--:--
   ---------------------------------------- 0.0/5.3 MB ? eta -:--:--
   ---------------------------------------- 0.0/5.3 MB ? eta -:--:--
   ---------------------------------------- 0.0/5.3 MB ? eta -:--:--
   ---------------------------------------- 0.0/5.3 MB ? eta -:--:--
   ---------------------------------------- 0.0/5.3 MB ? eta -:--:--
   - -------------------------------------- 0.3/5.3 MB ? eta -:--:--
   - -------------------------------------- 0.3/5.3 MB ? eta -:--:--
   - -------------------------------------- 0.3/5.3 MB ? eta -:--:--
   --- ------------------------------------ 0.5/5

In [6]:
%pip install importlib-resources

Note: you may need to restart the kernel to use updated packages.


In [7]:
import tensorflow_datasets as tfds

(train_ds, val_ds), ds_info = tfds.load(
    'tf_flowers',
    split=['train[:80%]', 'train[80%:]'],
    as_supervised=True,
    with_info=True
)


Dl Completed...: 0 url [00:00, ? url/s]

Dl Size...: 0 MiB [00:00, ? MiB/s]

Generating splits...:   0%|          | 0/1 [00:00<?, ? splits/s]

Generating train examples...: 0 examples [00:00, ? examples/s]

Shuffling C:\Users\khana\tensorflow_datasets\tf_flowers\incomplete.2C2QDA_3.0.1\tf_flowers-train.tfrecord-[0-9…

Dataset tf_flowers downloaded and prepared to C:\Users\khana\tensorflow_datasets\tf_flowers\3.0.1. Subsequent calls will reuse this data.


In [8]:
IMG_SIZE = 160

def preprocess(image, label):
    image = tf.image.resize(image, (IMG_SIZE, IMG_SIZE))  # resize all images to 160x160
    image = image / 255.0  # normalize pixel values between 0 and 1
    return image, label


In [9]:
train_ds = train_ds.map(preprocess).batch(32).shuffle(1000)
val_ds = val_ds.map(preprocess).batch(32)


In [10]:
base_model = MobileNetV2(input_shape=(IMG_SIZE, IMG_SIZE, 3),
                         include_top=False,
                         weights='imagenet')
base_model.trainable = False


9406464/9406464 ━━━━━━━━━━━━━━━━━━━━ 3s 0us/step


In [11]:
# This is the pre-trained MobileNetV2 (with all layers frozen).
# It extracts meaningful features from images.
x = base_model.output
# This flattens the output from the CNN to prepare it for classification.
x = GlobalAveragePooling2D()(x)
x = Dense(128, activation='relu')(x)
#  This is the classifier layer.
#It takes the features and classifies the image (e.g., "daisy" vs "not daisy").
predictions = Dense(5, activation='softmax')(x)





In [13]:
model = Model(inputs=base_model.input, outputs=predictions)


In [14]:
model.compile(optimizer='adam',
              loss='sparse_categorical_crossentropy',
              metrics=['accuracy'])


In [15]:
model.fit(train_ds, validation_data=val_ds, epochs=5)


Epoch 1/5


c:\Users\khana\anaconda3\Lib\site-packages\keras\src\trainers\epoch_iterator.py:74: UserWarning: `shuffle=True` was passed, but will be ignored since the data `x` was provided as a tf.data.Dataset. The Dataset is expected to already be shuffled (via `.shuffle(buffer_size)`).
  self.data_adapter = data_adapters.get_data_adapter(


92/92 ━━━━━━━━━━━━━━━━━━━━ 28s 222ms/step - accuracy: 0.7912 - loss: 0.5628 - val_accuracy: 0.8787 - val_loss: 0.3390
Epoch 2/5
92/92 ━━━━━━━━━━━━━━━━━━━━ 20s 213ms/step - accuracy: 0.9135 - loss: 0.2417 - val_accuracy: 0.8774 - val_loss: 0.3416
Epoch 3/5
92/92 ━━━━━━━━━━━━━━━━━━━━ 18s 198ms/step - accuracy: 0.9554 - loss: 0.1447 - val_accuracy: 0.8924 - val_loss: 0.3158
Epoch 4/5
92/92 ━━━━━━━━━━━━━━━━━━━━ 18s 191ms/step - accuracy: 0.9768 - loss: 0.0865 - val_accuracy: 0.8856 - val_loss: 0.3491
Epoch 5/5
92/92 ━━━━━━━━━━━━━━━━━━━━ 18s 189ms/step - accuracy: 0.9894 - loss: 0.0516 - val_accuracy: 0.8951 - val_loss: 0.3376
